In [ ]:
!pip install -q m2cgen

import numpy as np
import pandas as pd
from scipy.stats import mode
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import m2cgen as m2c
from google.colab import files

# LOAD DATA
print("Loading Data...")
try:
    df = pd.read_csv("CompleteDataSet.csv", low_memory=False)
except FileNotFoundError:
    print("ERROR: Upload 'CompleteDataSet.csv' first!")
    raise

# Header Fix
new_header = df.iloc[0]
df = df[1:].copy()
df.columns = new_header
df = df.reset_index(drop=True)

# Rename Columns
columns = ["TimeStamps"]
imu_names = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z","lux"]
for i in range(1, 7):
    for name in imu_names:
        columns.append(f"device{i}_{name}")
columns += ["Subject", "Activity", "Trial", "Tag"]
df.columns = columns

# PREPROCESS
def map_5_classes(tag):
    try:
        t = int(float(tag))
    except: return 2
    if 1 <= t <= 5: return 0      # Fall
    elif t in [6, 10]: return 1   # Active
    elif t in [7, 9]: return 2    # Stand
    elif t == 8: return 3         # Sit
    elif t == 11: return 4        # Lay
    return 2

df["label_class"] = df["Tag"].apply(map_5_classes)

# Fix Numeric
cols = ["device6_acc_x", "device6_acc_y", "device6_acc_z",
        "device6_gyro_x", "device6_gyro_y", "device6_gyro_z"]
for c in cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df.dropna(subset=cols, inplace=True)

# FEATURE ENGINEERING
WINDOW_SIZE = 50
STEP_SIZE   = 12

X_raw = df[cols].values
y_raw = df["label_class"].values

def extract_features(window):
    features = []
    # Reconstruct magnitudes for the window
    ax, ay, az = window[:,0], window[:,1], window[:,2]
    gx, gy, gz = window[:,3], window[:,4], window[:,5]
    amag = np.sqrt(ax**2+ay**2+az**2)
    gmag = np.sqrt(gx**2+gy**2+gz**2)
    full_window = np.column_stack((window, amag, gmag))

    for i in range(8):
        d = full_window[:, i]
        features.extend([
            np.mean(d), np.std(d), np.max(d), np.min(d),
            np.max(d)-np.min(d), np.sum(d**2)/len(d), np.sum(np.abs(np.diff(d)))
        ])
    return np.array(features)

print("Processing Windows (this may take a moment)...")
X_feat, y_labels = [], []

for start in range(0, len(X_raw) - WINDOW_SIZE, STEP_SIZE):
    end = start + WINDOW_SIZE
    window = X_raw[start:end]
    labels = y_raw[start:end]

    if 0 in labels: label = 0
    else: label = mode(labels, keepdims=True)[0][0]

    X_feat.append(extract_features(window))
    y_labels.append(label)

X_feat = np.array(X_feat)
y_labels = np.array(y_labels)

#TRAIN SINGLE DECISION TREE
print("Training Single Decision Tree...")
X_train, X_test, y_train, y_test = train_test_split(X_feat, y_labels, test_size=0.3, stratify=y_labels, random_state=42)

clf = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

# EVALUATION
target_names = ["FALL", "ACTIVE", "STAND", "SIT", "LAY"]
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\nTest Accuracy: {acc:.2%}")
print("-" * 30)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n--- Confusion Matrix (Rows=True, Cols=Pred) ---")
print(cm)

print("\n--- Accuracy Per Class ---")
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

for i, label in enumerate(target_names):
    print(f"{label:10s}: {cm_norm[i, i]:.2%} ({cm[i, i]}/{cm[i].sum()})")

# EXPORT
print("\nGenerating model.py...")
model_code = m2c.export_to_python(clf)

final_code = "# Single Decision Tree Model\n" + model_code

with open("model.py", "w") as f:
    f.write(final_code)

print("Downloading model.py...")
files.download("model.py")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 6.6 MB/s eta 0:00:00
Loading Data...
Processing Windows (this may take a moment)...
Training Single Decision Tree...

Test Accuracy: 65.41%
------------------------------

--- Confusion Matrix (Rows=True, Cols=Pred) ---
[[ 411   46   12    0   51]
 [ 400 1477   57   55   43]
 [  76   59 1166  132  174]
 [  65   65  764  386   67]
 [ 273   58  137   10 1371]]

--- Accuracy Per Class ---
FALL      : 79.04% (411/520)
ACTIVE    : 72.69% (1477/2032)
STAND     : 72.56% (1166/1607)
SIT       : 28.66% (386/1347)
LAY       : 74.15% (1371/1849)

Generating model.py...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>